# Designing and scheduling a Houston energy system

Choose capacity for a single-site system that supplies electricity and compressed hydrogen. Dispatch decisions follow the load and weather within each day. The study keeps the legacy coupling among renewable power, gas generation, electrolysis, compression, and electricity storage in a compact planning model.

## Historical day and assumptions

The table uses six four-hour means from July 15, 2019, in Houston local time. The wind speed and direct normal irradiance are from the Houston NSRDB series, and the COAST load comes from ERCOT. The load is scaled to a local peak of 45 MW. The PV and wind expressions below approximate the conversion from weather to capacity factors. The capital and operating costs are illustrative and expressed on the same daily basis.

In [1]:
from math import isclose

import pandas as pd
from energia import Model, Periods, Process, Resource, Storage
from gana import inf

profile = pd.DataFrame(
    [
        [0.0, 4.425, 13297.653],
        [0.5, 3.55, 13540.264],
        [149.875, 3.65, 15916.218],
        [175.125, 4.25, 18148.205],
        [61.125, 4.462, 18226.699],
        [0.25, 3.613, 16153.656],
    ],
    columns=["DNI (W/m²)", "Wind (m/s)", "ERCOT COAST load (MW)"],
    index=["00:00", "04:00", "08:00", "12:00", "16:00", "20:00"],
)
block_hours = 4
power_demand = (45 * profile.iloc[:, 2] / profile.iloc[:, 2].max()).tolist()
pv_availability = (profile.iloc[:, 0] / 1000).clip(0, 1).tolist()
wind_availability = (profile.iloc[:, 1] / 8).pow(3).clip(0, 1).tolist()
hydrogen_demand = 0.25
profile

,DNI (W/m²),Wind (m/s),ERCOT COAST load (MW)
00:00,0.000,4.425,13297.653
04:00,0.500,3.550,13540.264
08:00,149.875,3.650,15916.218
12:00,175.125,4.250,18148.205
16:00,61.125,4.462,18226.699
20:00,0.250,3.613,16153.656


## Capacity and dispatch

The conversion basis is one unit of output. A transport-ready tonne of hydrogen requires a tonne of hydrogen from the electrolyzer and 0.417 MWh of compression electricity. An operating level is an average rate over its four-hour block. Battery inventory is measured in equivalent blocks of discharge and converted to MWh for reporting.

In [2]:
m = Model("houston_design")
m.block = Periods()
m.day = len(profile) * m.block
m.declare(Resource, ["sun", "wind", "gas", "power", "hydrogen", "compressed_hydrogen"])
for resource in (m.sun, m.wind, m.gas):
    _ = resource.consume(m.block) >= 0

served_power = m.power.release(m.block)
served_hydrogen = m.compressed_hydrogen.release(m.block)
_ = served_power == power_demand
_ = served_hydrogen == [hydrogen_demand] * len(profile)

technologies = {
    "pv": (m.power == -m.sun, pv_availability, 180, 0),
    "wind_farm": (m.power == -m.wind, wind_availability, 170, 0),
    "gas_generator": (m.power == -0.108 * m.gas, 1, 100, 62),
    "electrolyzer": (m.hydrogen == -52 * m.power, 1, 90, 0),
    "compressor": (m.compressed_hydrogen == -m.hydrogen - 0.417 * m.power, 1, 25, 0),
}
cost = 0
for name, (conversion, availability, capacity_cost, dispatch_cost) in technologies.items():
    plant = Process(conversion)
    setattr(m, name, plant)
    capacity = plant.capacity(m.day)
    dispatch = plant.operate(m.block)
    _ = capacity <= 200
    _ = dispatch <= availability
    plant.locate()
    cost += capacity_cost * capacity.V()[0]
    if dispatch_cost:
        cost += block_hours * dispatch_cost * sum(dispatch.V())

m.battery = Storage(m.power == 0.9)
_ = m.battery.capacity(m.day) <= 50
m.battery.locate()
cost += 60 * m.battery.capacity(m.day).V()[0]

m.program.daily_cost = inf(cost)
m.opt()

## Results

In [3]:
assert all(isclose(x, y, abs_tol=1e-6) for x, y in zip(served_power.output(aslist=True), power_demand, strict=True))
assert isclose(block_hours * sum(served_hydrogen.output(aslist=True)), 6)
print(f"Daily cost: {m.program.active_objective.X:,.2f}")
pd.DataFrame({
    "Load (MW)": power_demand,
    "PV (MW)": m.pv.operate(m.block).output(aslist=True),
    "Wind (MW)": m.wind_farm.operate(m.block).output(aslist=True),
    "Gas generator (MW)": m.gas_generator.operate(m.block).output(aslist=True),
    "Battery inventory (MWh)": [block_hours * x for x in m.battery.inventory(m.block).output(aslist=True)],
}, index=profile.index).round(2)

Daily cost: 77,267.90


,Load (MW),PV (MW),Wind (MW),Gas generator (MW),Battery inventory (MWh)
00:00,32.83,-0.0,33.85,13.38,5.15
04:00,33.43,-0.0,17.48,29.06,5.15
08:00,39.30,-0.0,18.99,33.40,5.15
12:00,44.81,-0.0,29.99,27.92,5.15
16:00,45.00,-0.0,34.70,23.40,5.15
20:00,39.88,-0.0,18.42,33.40,0.00
